In [1]:
import torch

In [2]:
import numpy , pandas

In [3]:
!git clone https://github.com/openai/shap-e.git

Cloning into 'shap-e'...
remote: Enumerating objects: 336, done.
remote: Counting objects: 100% (260/260), done.
remote: Compressing objects: 100% (241/241), done.
remote: Total 336 (delta 41), reused 218 (delta 18), pack-reused 76 (from 1)
Receiving objects: 100% (336/336), 11.72 MiB | 13.75 MiB/s, done.
Resolving deltas: 100% (44/44), done.


In [4]:
%cd shap-e

/content/shap-e


In [5]:
!pwd

/content/shap-e


In [6]:
!pip install -e .

Obtaining file:///content/shap-e
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-install-yco5e039/clip_3aa39155b05f4fa5ac30985073b73948
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-install-yco5e039/clip_3aa39155b05f4fa5ac30985073b73948
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.5 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369489 sha256=27c5c5194b2249d6ee66b2af697da5bf5379ce79256f54622b474799fefe08b1
  Stored in directory:

In [7]:
import torch

from shap_e.diffusion.sample import sample_latents
from shap_e.diffusion.gaussian_diffusion import diffusion_from_config
from shap_e.models.download import load_model, load_config
from shap_e.util.notebooks import create_pan_cameras, decode_latent_images, gif_widget
from shap_e.util.image_util import load_image

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [9]:
device

device(type='cuda')

In [10]:
torch.cuda.device_count()

1

In [11]:
torch.cuda.current_device()

0

In [12]:
torch.cuda.get_device_name(0)

'Tesla T4'

In [13]:
xm = load_model('transmitter', device=device)
model = load_model('image300M', device=device)
diffusion = diffusion_from_config(load_config('diffusion'))


/content/shap-e/shap_e/models/nn/checkpoint.py:31: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/content/shap-e/shap_e/models/nn/checkpoint.py:43: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/content/shap-e/shap_e/models/nn/checkpoint.py:61: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/content/shap-e/shap_e/models/nn/checkpoint.py:86: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd


  0%|          | 0.00/1.78G [00:00<?, ?iB/s]

/content/shap-e/shap_e/models/download.py:136: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(path, map_location=device)
100%|██████████████████████████████

  0%|          | 0.00/1.26G [00:00<?, ?iB/s]

/content/shap-e/shap_e/models/download.py:136: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(path, map_location=device)


In [ ]:
batch_size = 4
guidance_scale = 3.0

# To get the best result, you should remove the background and show only the object of interest to the model.
image = load_image("/content/beside_table_modern.jpg")

latents = sample_latents(
    batch_size=batch_size,
    model=model,
    diffusion=diffusion,
    guidance_scale=guidance_scale,
    model_kwargs=dict(images=[image] * batch_size),
    progress=True,
    clip_denoised=True,
    use_fp16=True,
    use_karras=True,
    karras_steps=64,
    sigma_min=1e-3,
    sigma_max=160,
    s_churn=0,
)

  0%|          | 0/64 [00:00<?, ?it/s]

changing scaling factor to 15

In [ ]:
batch_size = 4
guidance_scale = 3.5

# To get the best result, you should remove the background and show only the object of interest to the model.
image = load_image("/content/beside_table_modern.jpg")

latents = sample_latents(
    batch_size=batch_size,
    model=model,
    diffusion=diffusion,
    guidance_scale=guidance_scale,
    model_kwargs=dict(images=[image] * batch_size),
    progress=True,
    clip_denoised=True,
    use_fp16=True,
    use_karras=True,
    karras_steps=64,
    sigma_min=1e-3,
    sigma_max=160,
    s_churn=0,
)

  0%|          | 0/64 [00:00<?, ?it/s]

In [ ]:
render_mode = 'nerf' # you can change this to 'stf' for mesh rendering
size = 256 # this is the size of the renders; higher values take longer to render.

cameras = create_pan_cameras(size, device)
for i, latent in enumerate(latents):
    images = decode_latent_images(xm, latent, cameras, rendering_mode=render_mode)
    display(gif_widget(images))


HTML(value='<img src="data:image/gif;base64,R0lGODlhAAEAAYcAAOLo7Nri6d3f1vHbke7Yj+3Ui8nT2ufOh+THg+PCfuG+et3Aft…

HTML(value='<img src="data:image/gif;base64,R0lGODlhAAEAAYcAAOC4h+Kyf+Owfd2vfteyhNWxhNWwg9avgOate+CtfN2tfNWuf+…

HTML(value='<img src="data:image/gif;base64,R0lGODlhAAEAAYcAAOvr6uvq6urq6unq6urp6urp6enp6enp6Ojp6eno6eno6Ojo6e…

HTML(value='<img src="data:image/gif;base64,R0lGODlhAAEAAYcAAN+rZd6pY92oYd2lXtqnY9qlY9KlZ9iiYNKhY8yjacyiaMyhZ8…

for converting chair to 3d model

In [ ]:
batch_size = 4
guidance_scale = 3

# To get the best result, you should remove the background and show only the object of interest to the model.
image = load_image("/content/chairs_comftable_bedroom.jpg")

latents = sample_latents(
    batch_size=batch_size,
    model=model,
    diffusion=diffusion,
    guidance_scale=guidance_scale,
    model_kwargs=dict(images=[image] * batch_size),
    progress=True,
    clip_denoised=True,
    use_fp16=True,
    use_karras=True,
    karras_steps=64,
    sigma_min=1e-3,
    sigma_max=160,
    s_churn=0,
)

  0%|          | 0/64 [00:00<?, ?it/s]

changing scaling factor to 15

In [ ]:
render_mode = 'nerf' # you can change this to 'stf' for mesh rendering
size = 256 # this is the size of the renders; higher values take longer to render.

cameras = create_pan_cameras(size, device)
for i, latent in enumerate(latents):
    images = decode_latent_images(xm, latent, cameras, rendering_mode=render_mode)
    display(gif_widget(images))


HTML(value='<img src="data:image/gif;base64,R0lGODlhAAEAAYcAAJmdoZebn5aanpWZnZSYnJWXnJOXmpSWnJOWm5OWmZKWmZOVnJ…

HTML(value='<img src="data:image/gif;base64,R0lGODlhAAEAAYcAAK+0t6musqissKWssKapraSpraOprqKorKSnqqKnq6KmqqGnrK…

HTML(value='<img src="data:image/gif;base64,R0lGODlhAAEAAYcAAKuvtKKmq5+jqJ6ip52ipp6hpp2hppyhpp2gpZygppygpJugpJ…

HTML(value='<img src="data:image/gif;base64,R0lGODlhAAEAAYcAAK+1u62zuKuxtqmwtaivtKius6etsqatsqWssaWrr6Srr6Oqr6…

for converting chair to 3d model

for 3.5 guidance scale

In [ ]:
batch_size = 4
guidance_scale = 3.5

# To get the best result, you should remove the background and show only the object of interest to the model.
image = load_image("/content/chairs_comftable_bedroom.jpg")

latents = sample_latents(
    batch_size=batch_size,
    model=model,
    diffusion=diffusion,
    guidance_scale=guidance_scale,
    model_kwargs=dict(images=[image] * batch_size),
    progress=True,
    clip_denoised=True,
    use_fp16=True,
    use_karras=True,
    karras_steps=64,
    sigma_min=1e-3,
    sigma_max=160,
    s_churn=0,
)

  0%|          | 0/64 [00:00<?, ?it/s]

In [ ]:
render_mode = 'nerf' # you can change this to 'stf' for mesh rendering
size = 256 # this is the size of the renders; higher values take longer to render.

cameras = create_pan_cameras(size, device)
for i, latent in enumerate(latents):
    images = decode_latent_images(xm, latent, cameras, rendering_mode=render_mode)
    display(gif_widget(images))


HTML(value='<img src="data:image/gif;base64,R0lGODlhAAEAAYcAAKqpr6ioraenrKemq6Wlq6WkqaSkqqOjqKKiqKGhpqCgppygpp…

HTML(value='<img src="data:image/gif;base64,R0lGODlhAAEAAYcAALe+xrO6wrC0u6uyvK2usquqrqmpq6WrsqSoraemqaWlqKKmq6…

# **Lightshade**

In [16]:
batch_size = 8
guidance_scale = 3.0

# To get the best result, you should remove the background and show only the object of interest to the model.
image = load_image("/content/lightshade02.jpg")

latents = sample_latents(
    batch_size=batch_size,
    model=model,
    diffusion=diffusion,
    guidance_scale=guidance_scale,
    model_kwargs=dict(images=[image] * batch_size),
    progress=True,
    clip_denoised=True,
    use_fp16=True,
    use_karras=True,
    karras_steps=64,
    sigma_min=1e-3,
    sigma_max=160,
    s_churn=0,
)

  0%|          | 0/64 [00:00<?, ?it/s]

In [17]:
render_mode = 'nerf' # you can change this to 'stf' for mesh rendering
size = 256 # this is the size of the renders; higher values take longer to render.

cameras = create_pan_cameras(size, device)
for i, latent in enumerate(latents):
    images = decode_latent_images(xm, latent, cameras, rendering_mode=render_mode)
    display(gif_widget(images))


HTML(value='<img src="data:image/gif;base64,R0lGODlhAAEAAYcAANXVztXUztTUzdPTzNPSzNLSy9LRytHRydHQydDPyM/Px87Nxc…

HTML(value='<img src="data:image/gif;base64,R0lGODlhAAEAAYcAANva1tnZ1dnY1NjY09jX0tfX0dfW0dbW0NbVz9XVz9bUztXUzt…

HTML(value='<img src="data:image/gif;base64,R0lGODlhAAEAAYcAAODf3t/f3t/f3d/f3N/e3N7f297e293d2tzd2Nzc2Nvc2Nvc19…

HTML(value='<img src="data:image/gif;base64,R0lGODlhAAEAAYcAANnZ0tnY0tjY0tfX0NbWztbVzdTTy8/OxcvKwMnIv8nHvsjHv8…

KeyboardInterrupt: 